In [4]:
import duckdb
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

print("1. Connecting to DuckDB Lakehouse...")
# The database file was created one folder up from the pipeline directory
con = duckdb.connect('../supply_chain.duckdb')

# Extract our Gold Layer table
df = con.execute("SELECT * FROM fct_orders").df()
print(f"Loaded {len(df)} historical orders.")

print("\n2. Feature Engineering...")
# Extract time-based features
df['order_hour'] = df['order_placed_at'].dt.hour
df['order_day_of_week'] = df['order_placed_at'].dt.dayofweek

# One-hot encode categorical variables (like Country and Category)
df_encoded = pd.get_dummies(df, columns=['product_category', 'destination_country'])

# Define our inputs (X) and our target to predict (y)
drop_cols = ['order_id', 'customer_id', 'product_id', 'order_status', 'order_placed_at', 'is_delayed']
feature_cols = [col for col in df_encoded.columns if col not in drop_cols]

X = df_encoded[feature_cols]
y = df_encoded['is_delayed']

print("\n3. Splitting Data & Training XGBoost Model...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the Extreme Gradient Boosting Classifier
model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

print("\n4. Model Evaluation:")
preds = model.predict(X_test)
print(classification_report(y_test, preds))

print("\n5. Saving Model Artifact for Production...")
with open('delay_prediction_model.pkl', 'wb') as f:
    pickle.dump(model, f)
    
# Save the list of feature columns so our FastAPI/AI Agent knows exactly what the model expects
with open('model_features.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)
    
print("SUCCESS: Model and features saved to the notebooks directory!")

1. Connecting to DuckDB Lakehouse...
Loaded 1976 historical orders.

2. Feature Engineering...

3. Splitting Data & Training XGBoost Model...

4. Model Evaluation:
              precision    recall  f1-score   support

           0       0.76      0.98      0.85       301
           1       0.00      0.00      0.00        95

    accuracy                           0.74       396
   macro avg       0.38      0.49      0.43       396
weighted avg       0.57      0.74      0.65       396


5. Saving Model Artifact for Production...
SUCCESS: Model and features saved to the notebooks directory!
